# Solar filament CV: train, evaluate, and export

**Kaggle GPU pipeline:** grayscale observations + COCO annotations -> day-grouped train/tuning/holdout -> pretrained U-Net -> threshold/area calibration -> individual masks -> full-resolution COCO RLE CSV. An intensity/contrast CV baseline is evaluated on the same splits. Run on Kaggle with the competition data attached, a GPU accelerator, and Internet enabled for installation and pretrained weights. Default: 20 epochs; this is a starting configuration, not a demonstrated optimal model.

## Competition and paper requirements verified November 12, 2026
- [Live overview](https://www.kaggle.com/competitions/filament-segmentation-2026/overview): leaderboard **Panoptic Quality (PQ)**, not the anomaly papers' F1. Final judging is **70% quantitative, 30% qualitative**. F1 and PQ are objective given labels; annotator variation and the qualitative rubric introduce separate uncertainty/judgment.
- CSV: `filament_id,segmentation_rle`, one row per filament. IDs preserve the image stem and add unique suffixes. Counts are compressed COCO RLE for **2048 x 2048** masks, not the example notebooks' 512 x 512 output. No invented empty-filament fallback.
- Final evaluation needs a **four-page PDF**, [organizer template](https://www.overleaf.com/read/vjztpvrnbrdh#43fbbe), public Git repository, pinned packages, this notebook and saved weights for inference without retraining, plus the [final submission form](https://docs.google.com/forms/d/e/1FAIpQLSfyrM4dhc8QzxR90PtF6fb018KPf0S0Cbf5hPqMuUognmzQsA/viewform?usp=dialog). The page states November 15, 2026, 11:59 pm CDT; confirm that stated timezone because November normally uses CST.
- Old Dice wording remains on the page. Local class-agnostic PQ below is a transparent diagnostic, not a reproduction of all private server checks. Compare with the [organizer's self-evaluation notebook](https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook).

## Research and model selection
The ESAIR 2026 poster describes **U-Net + MIoU -> geometry/spine grouping -> ConvNeXt chirality classification**. Supervised SVM and unsupervised one-class SVM in the 2025 papers classify **image quality**, not filament pixels. One-class SVM is not supervised binary SVM. Here, all chirality categories are merged into binary filament foreground; classification is not needed for the CSV.

[Segmentation Models PyTorch](https://smp.readthedocs.io/en/latest/models.html) supplies a reusable single-channel U-Net with pretrained ResNet encoders. [YOLOX](https://github.com/Megvii-BaseDetection/YOLOX) is a box detector, not a mask model; [Mask R-CNN](https://docs.pytorch.org/vision/stable/models/mask_rcnn.html) is a valid future instance-segmentation comparison with a different training interface. U-Net is a justified baseline, not a proven winner. This implementation uses Dice+BCE and connected components, **not a claimed reproduction of MIoU or the poster's unpublished grouping**.

Sources: [SoftwareX 2025](https://doi.org/10.1016/j.softx.2025.102120), [ICDMW 2025](https://doi.org/10.1109/ICDMW69685.2025.00235), supplied [ESAIR poster](./ESAIRPoster/GRF_Poster_04.2026.pdf), and [research summaries](./solar_filament_data_and_research.ipynb). Quality-control F1 scores are not segmentation results. Kaggle upload is off by default; the paper/form submission is separate.

## 1. Import libraries and configure the experiment

The setup cell installs only the segmentation dependencies, using the selected notebook kernel. Keep Kaggle's matched PyTorch/torchvision GPU installation. Exact installed versions are recorded after execution. For offline reproduction, install recorded dependencies and set `RUN_TRAINING=False`, `RUN_CALIBRATION=False` and `CHECKPOINT_INPUT_DIR` to the attached saved experiment folder; weights and selected parameters are loaded without downloading ImageNet weights.

If you change the architecture or split, use a new `RUN_NAME`. Never treat a short debugging run as publication evidence.

In [ ]:
%pip install -q 'segmentation-models-pytorch>=0.3.3,<0.7' pycocotools scipy


In [ ]:
import csv
import hashlib
import importlib.metadata
import itertools
import json
import random
import shutil
import subprocess
import sys
import time
from collections import Counter, defaultdict
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy.ndimage as ndi
from scipy.optimize import linear_sum_assignment
from PIL import Image
import pycocotools.mask as mask_utils
import segmentation_models_pytorch as smp
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

CFG = dict(seed=2026, image_size=512, batch_size=8, epochs=20,
           learning_rate=3e-4, encoder='resnet18', architecture='Unet',
           pretrained=True, workers=0, tune_fraction=0.15, holdout_fraction=0.15,
           tune_image_limit=32, thresholds=[0.35, 0.5, 0.65],
           min_areas=[64, 256], contrast_thresholds=[0.025, 0.04, 0.06])
RUN_NAME = 'unet_resnet18_gray_512_seed2026'
RUN_TRAINING = True
RUN_CALIBRATION = True
SMOKE_TEST = False
SUBMIT_TO_KAGGLE = False
DATA_ROOT_OVERRIDE = None
CHECKPOINT_INPUT_DIR = None  # Path to an attached experiment folder for inference-only reproduction.
OUTPUT_DIR = (Path('/kaggle/working') if Path('/kaggle/working').is_dir()
              else Path.cwd() / 'cv_outputs') / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR = Path(CHECKPOINT_INPUT_DIR) if CHECKPOINT_INPUT_DIR else OUTPUT_DIR
CHECKPOINT_PATH = INPUT_DIR / 'best_model.pt'
COMPETITION = 'filament-segmentation-2026'
DATASET_NAME = 'MAGFiLO_1.0_Kaggle_2026'
ANNOTATION_NAME = 'MAGFiLO_1.0_Annotations_kaggle2026_train.json'
FULL_SHAPE = (2048, 2048)

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['seed'])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cpu':
    print('WARNING: CPU selected. Enable a Kaggle GPU before full training.')
if CFG['image_size'] % 32 or CFG['batch_size'] < 2 or CFG['epochs'] < 1:
    raise ValueError('Use image_size divisible by 32, batch_size >= 2, and epochs >= 1.')
print('Device:', DEVICE, 'outputs:', OUTPUT_DIR)

def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False))


## 2. Load training data and prevent annotation leakage

Attach the competition on Kaggle. Local data from the loading notebook is also supported; this notebook does not silently download 750 MB. Multiple annotator-specific image IDs refer to the same JPEG. Each annotation set remains a separate supervised target; splits are grouped by **observing day** so a physical image and same-day observations cannot leak across partitions. Training targets are unions of filaments **within one annotation set**, never a union across annotators.

We use about 70% of days for fitting, 15% for tuning/checkpoint selection, and 15% as an untouched holdout. Calibration can use up to 32 tuning images to bound runtime; holdout evaluation uses all holdout images. Holdout results must not be used to retune. Image resizing is an explicit first baseline limitation; thin structures may disappear. A tiled/full-resolution ablation is future work.

In [ ]:
candidates = ([Path(DATA_ROOT_OVERRIDE)] if DATA_ROOT_OVERRIDE else [
    Path('/kaggle/input/competitions') / COMPETITION / DATASET_NAME,
    Path('/kaggle/input') / COMPETITION / DATASET_NAME,
    Path.cwd() / 'data' / DATASET_NAME, Path.cwd() / DATASET_NAME])
if DATA_ROOT_OVERRIDE is None and Path('/kaggle/input').is_dir():
    candidates += sorted(Path('/kaggle/input').glob(f'*/{DATASET_NAME}'))
    candidates += sorted(Path('/kaggle/input').glob(f'*/*/{DATASET_NAME}'))
valid_roots = [p for p in candidates if (p / 'train' / ANNOTATION_NAME).is_file()
               and (p / 'test' / 'test_images').is_dir()]
if not valid_roots:
    raise FileNotFoundError('Attach the competition data or set DATA_ROOT_OVERRIDE to its dataset root.')
DATA_ROOT = valid_roots[0]
train_paths = sorted((DATA_ROOT / 'train' / 'train_images').glob('*.jpeg'))
test_paths = sorted((DATA_ROOT / 'test' / 'test_images').glob('*.jpeg'))
if not train_paths or not test_paths:
    raise FileNotFoundError('Train and test must both contain JPEG files.')
with (DATA_ROOT / 'train' / ANNOTATION_NAME).open() as stream:
    coco = json.load(stream)
records = coco['images']
if len({r['id'] for r in records}) != len(records):
    raise ValueError('Duplicate COCO image IDs.')
records_by_name = defaultdict(list)
annotations_by_id = defaultdict(list)
for record in records:
    if (record['height'], record['width']) != FULL_SHAPE:
        raise ValueError(f'Unexpected dimensions for {record["id"]}')
    records_by_name[record['file_name']].append(record)
for ann in coco['annotations']:
    if ann.get('iscrowd', 0) != 0:
        raise ValueError('Local PQ assumes individual non-crowd filaments.')
    annotations_by_id[ann['image_id']].append(ann)
known_ids = {r['id'] for r in records}
if set(annotations_by_id) - known_ids:
    raise ValueError('Orphan annotation image IDs.')
if len({a['id'] for a in coco['annotations']}) != len(coco['annotations']):
    raise ValueError('Duplicate filament annotation IDs.')
train_lookup = {p.name: p for p in train_paths}
if set(records_by_name) - set(train_lookup):
    raise FileNotFoundError('Annotations reference missing training files.')
if set(train_lookup) & {p.name for p in test_paths}:
    raise ValueError('Train/test filenames overlap.')
days = sorted({name[:8] for name in records_by_name})
if len(days) < 6:
    raise ValueError('Not enough observing days for three nonempty grouped splits.')
random.Random(CFG['seed']).shuffle(days)
n_tune = max(1, round(len(days) * CFG['tune_fraction']))
n_holdout = max(1, round(len(days) * CFG['holdout_fraction']))
if n_tune + n_holdout >= len(days):
    raise ValueError('Tuning and holdout fractions leave no training days.')
day_sets = dict(tune=set(days[:n_tune]), holdout=set(days[n_tune:n_tune+n_holdout]),
                train=set(days[n_tune+n_holdout:]))
split_names = {split: sorted(n for n in records_by_name if n[:8] in selected)
               for split, selected in day_sets.items()}
split_records = {split: [r for name in names for r in records_by_name[name]]
                 for split, names in split_names.items()}
for first, second in itertools.combinations(day_sets, 2):
    assert not day_sets[first] & day_sets[second]
annotation_sha256 = hashlib.sha256((DATA_ROOT / 'train' / ANNOTATION_NAME).read_bytes()).hexdigest()
manifest = dict(seed=CFG['seed'], grouping='observation-day', annotation_sha256=annotation_sha256,
                policy='separate annotator targets; union within each target', splits=split_names)
write_json(OUTPUT_DIR / 'split_manifest.json', manifest)
print('Data root:', DATA_ROOT)
print('JPEGs:', len(train_paths), 'train,', len(test_paths), 'test')
print('Annotation sets:', len(records), 'filaments:', len(coco['annotations']))
print('Categories:', Counter(a['category_id'] for a in coco['annotations']))
print('Splits:', {s: (len(split_names[s]), len(split_records[s])) for s in split_names})


In [ ]:
@lru_cache(maxsize=128)
def small_image(filename):
    with Image.open(train_lookup[filename]) as image:
        if image.mode != 'L' or image.size != (2048, 2048):
            raise ValueError(f'{filename}: expected 2048 x 2048 grayscale JPEG.')
        image = image.resize((CFG['image_size'], CFG['image_size']), Image.Resampling.BILINEAR)
        return np.asarray(image, dtype=np.float32) / 255.0

def ann_rle(annotation):
    segmentation = annotation['segmentation']
    if isinstance(segmentation, list):
        if not segmentation or any(len(p) < 6 or len(p) % 2 for p in segmentation):
            raise ValueError(f'Invalid polygon: {annotation["id"]}')
        rle = mask_utils.merge(mask_utils.frPyObjects(segmentation, *FULL_SHAPE))
    elif isinstance(segmentation, dict):
        if list(segmentation['size']) != list(FULL_SHAPE):
            raise ValueError('RLE dimensions do not match the image.')
        if isinstance(segmentation['counts'], list):
            rle = mask_utils.frPyObjects(segmentation, *FULL_SHAPE)
        else:
            counts = segmentation['counts']
            rle = dict(size=list(FULL_SHAPE), counts=counts.encode('ascii') if isinstance(counts, str) else counts)
    else:
        raise TypeError(f'Unsupported segmentation: {annotation["id"]}')
    if mask_utils.area(rle) <= 0:
        raise ValueError(f'Empty ground-truth filament: {annotation["id"]}')
    return rle

@lru_cache(maxsize=512)
def gt_rles(image_id):
    return [ann_rle(a) for a in annotations_by_id[image_id]]

def union_mask(rles):
    if not rles:
        return np.zeros(FULL_SHAPE, dtype=np.uint8)
    return mask_utils.decode(mask_utils.merge(rles))

@lru_cache(maxsize=128)
def small_target(image_id):
    full_mask = union_mask(gt_rles(image_id))
    resized = Image.fromarray(full_mask).resize((CFG['image_size'], CFG['image_size']), Image.Resampling.NEAREST)
    return np.asarray(resized, dtype=np.float32)

class FilamentDataset(Dataset):
    def __init__(self, entries, augment=False):
        self.entries = entries
        self.augment = augment

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, index):
        record = self.entries[index]
        image = small_image(record['file_name'])
        target = small_target(record['id'])
        if self.augment:
            turns = random.randrange(4)
            image, target = np.rot90(image, turns), np.rot90(target, turns)
            if random.random() < 0.5:
                image, target = np.fliplr(image), np.fliplr(target)
        x = torch.from_numpy(np.ascontiguousarray((image - 0.5) / 0.5))[None]
        y = torch.from_numpy(np.ascontiguousarray(target))[None]
        return x, y

example = split_records['train'][0]
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(small_image(example['file_name']), cmap='gray', vmin=0, vmax=1)
axes[1].imshow(small_image(example['file_name']), cmap='gray', vmin=0, vmax=1)
axes[1].contour(small_target(example['id']), levels=[0.5], colors=['red'], linewidths=0.5)
axes[0].set_title(example['file_name']); axes[1].set_title('One annotator target (red)')
for axis in axes: axis.axis('off')
fig.savefig(OUTPUT_DIR / 'training_annotation.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Train the supervised model, then tune only on calibration data

U-Net/ResNet18 predicts one filament probability per pixel, with Dice+BCE loss and paired geometric augmentation. Inputs are normalized grayscale in [-1, 1]; SMP adapts pretrained encoder weights to one input channel. Change `architecture` to `FPN` for an established-library architecture ablation, using a new run name. Reduce batch size if GPU memory is insufficient.

Checkpoint selection uses tuning-set loss; threshold and minimum original-resolution instance area are selected by local PQ. The independent holdout is evaluated afterward. We do **not** retrain on holdout data, so the saved weights exactly reproduce the reported predictions. Twenty epochs and 512-pixel inputs are baseline settings requiring convergence and resolution studies.

In [ ]:
def build_model(config, pretrained=False):
    constructors = {'Unet': smp.Unet, 'FPN': smp.FPN}
    if config['architecture'] not in constructors:
        raise ValueError('Supported architectures: Unet, FPN.')
    return constructors[config['architecture']](encoder_name=config['encoder'],
        encoder_weights='imagenet' if pretrained else None, in_channels=1, classes=1, activation=None)

def segmentation_loss(logits, target):
    probabilities = logits.sigmoid()
    intersection = (probabilities * target).sum(dim=(1, 2, 3))
    denominator = probabilities.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    dice = 1 - ((2 * intersection + 1) / (denominator + 1)).mean()
    return 0.5 * F.binary_cross_entropy_with_logits(logits, target) + 0.5 * dice

if RUN_TRAINING:
    if CHECKPOINT_INPUT_DIR is not None:
        raise ValueError('Use a new output run to train; do not overwrite attached inference weights.')
    if CHECKPOINT_PATH.exists():
        raise FileExistsError('Existing checkpoint: set RUN_TRAINING=False or choose a new RUN_NAME.')
    model = build_model(CFG, pretrained=CFG['pretrained']).to(DEVICE)
    train_loader = DataLoader(FilamentDataset(split_records['train'], augment=True),
        batch_size=CFG['batch_size'], shuffle=True, num_workers=CFG['workers'],
        drop_last=len(split_records['train']) % CFG['batch_size'] == 1)
    tune_loader = DataLoader(FilamentDataset(split_records['tune']),
        batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['workers'])
    if len(train_loader) == 0 or len(tune_loader) == 0:
        raise ValueError('No batches: increase training sample count or lower batch size.')
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['learning_rate'])
    best_loss, history = float('inf'), []
    started = time.perf_counter()
    for epoch in range(CFG['epochs']):
        totals = {}
        for phase, loader in [('train', train_loader), ('tune', tune_loader)]:
            model.train(phase == 'train')
            total, count = 0.0, 0
            with torch.set_grad_enabled(phase == 'train'):
                for images, targets in tqdm(loader, desc=f'Epoch {epoch+1} {phase}'):
                    images, targets = images.to(DEVICE), targets.to(DEVICE)
                    if phase == 'train': optimizer.zero_grad(set_to_none=True)
                    loss = segmentation_loss(model(images), targets)
                    if not torch.isfinite(loss):
                        raise FloatingPointError(f'Nonfinite loss in {phase}.')
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                    total += loss.item() * len(images); count += len(images)
            totals[phase] = total / count
        history.append(dict(epoch=epoch+1, train_loss=totals['train'], tune_loss=totals['tune']))
        print(history[-1])
        if totals['tune'] < best_loss:
            best_loss = totals['tune']
            torch.save(dict(state_dict={k: v.detach().cpu() for k, v in model.state_dict().items()},
                            config=dict(CFG), epoch=epoch+1, tune_loss=best_loss,
                            annotation_sha256=annotation_sha256, smoke_test=SMOKE_TEST), CHECKPOINT_PATH)
    write_json(OUTPUT_DIR / 'training_history.json', history)
    write_json(OUTPUT_DIR / 'training_runtime.json', dict(seconds=time.perf_counter()-started,
        device=str(DEVICE), gpu=torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None))

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError('Train first or attach the saved best_model.pt experiment.')
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
if checkpoint['annotation_sha256'] != annotation_sha256:
    raise ValueError('Checkpoint was trained against a different annotation file.')
if checkpoint['smoke_test'] != SMOKE_TEST:
    raise ValueError('Smoke-test checkpoint cannot be used for a competition run.')
if checkpoint['config'] != CFG:
    raise ValueError('Configuration differs from checkpoint; restore its config for reproduction.')
model = build_model(checkpoint['config'], pretrained=False).to(DEVICE)
model.load_state_dict(checkpoint['state_dict'], strict=True)
model.eval()
CHECKPOINT_SHA256 = hashlib.sha256(CHECKPOINT_PATH.read_bytes()).hexdigest()
print('Loaded best epoch:', checkpoint['epoch'], 'tuning loss:', checkpoint['tune_loss'])


In [ ]:
@torch.inference_mode()
def predict_probability(path):
    with Image.open(path) as image:
        if image.mode != 'L' or image.size != (2048, 2048):
            raise ValueError(f'{path.name}: invalid image mode or dimensions.')
        image = image.resize((CFG['image_size'], CFG['image_size']), Image.Resampling.BILINEAR)
        array = np.asarray(image, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(np.ascontiguousarray((array - 0.5) / 0.5))[None, None].to(DEVICE)
    probability = model(tensor).sigmoid()[0, 0].cpu().numpy()
    if not np.isfinite(probability).all():
        raise FloatingPointError(f'Nonfinite prediction: {path.name}')
    return probability

def full_probability(probability):
    resized = Image.fromarray(np.asarray(probability, dtype=np.float32)).resize(
        (2048, 2048), Image.Resampling.BILINEAR)
    return np.asarray(resized, dtype=np.float32)

def contrast_foreground(filename, threshold):
    image = small_image(filename)
    disk = ndi.binary_erosion(image > 0.08, iterations=max(1, CFG['image_size']//128))
    background = ndi.gaussian_filter(image, sigma=CFG['image_size']/32)
    return disk & ((background - image) > threshold)

def instance_rles(probability, threshold, min_area):
    if min_area < 1 or not 0 <= threshold <= 1:
        raise ValueError('Invalid threshold or minimum area.')
    binary = full_probability(probability) >= threshold
    labels, count = ndi.label(binary, structure=np.ones((3, 3), dtype=np.uint8))
    areas = np.bincount(labels.ravel())
    result = []
    for label in range(1, count+1):
        if areas[label] >= min_area:
            mask = np.asfortranarray(labels == label, dtype=np.uint8)
            result.append(mask_utils.encode(mask))
    return result

def local_scores(predicted, truth):
    n_pred, n_true = len(predicted), len(truth)
    overlaps = (mask_utils.iou(predicted, truth, [0]*n_true)
                if n_pred and n_true else np.zeros((n_pred, n_true)))
    matched = []
    if n_pred and n_true:
        valid = overlaps > 0.5
        # Maximum cardinality first, then overlap; greedy matching can lose true positives.
        rewards = valid * (1 + overlaps / (min(n_pred, n_true) + 1))
        rows, columns = linear_sum_assignment(-rewards)
        matched = [float(overlaps[r, c]) for r, c in zip(rows, columns) if valid[r, c]]
    tp = len(matched); fp = n_pred-tp; fn = n_true-tp
    denominator = tp + 0.5*fp + 0.5*fn
    pq = sum(matched)/denominator if denominator else 1.0
    pred_union, true_union = union_mask(predicted).astype(bool), union_mask(truth).astype(bool)
    intersection = np.count_nonzero(pred_union & true_union)
    union = np.count_nonzero(pred_union | true_union)
    pixels = np.count_nonzero(pred_union) + np.count_nonzero(true_union)
    return dict(pq=pq, dice=2*intersection/pixels if pixels else 1.0,
                iou=intersection/union if union else 1.0, tp=tp, fp=fp, fn=fn,
                split_relations=int(np.count_nonzero((overlaps > 0.1).sum(axis=0) > 1)),
                merge_relations=int(np.count_nonzero((overlaps > 0.1).sum(axis=1) > 1)))

def evaluate(names, parameters, probability_cache=None):
    rows = []
    for filename in names:
        if parameters['method'] == 'unet':
            probability = (probability_cache[filename] if probability_cache is not None
                           else predict_probability(train_lookup[filename]))
            threshold = parameters['threshold']
        elif parameters['method'] == 'contrast':
            probability = contrast_foreground(filename, parameters['contrast_threshold']).astype(np.float32)
            threshold = 0.5
        else:
            raise ValueError('Unknown segmentation method.')
        predicted = instance_rles(probability, threshold, parameters['min_area'])
        for record in records_by_name[filename]:
            rows.append(dict(filename=filename, image_id=record['id'],
                             **local_scores(predicted, gt_rles(record['id']))))
    if not rows:
        raise ValueError('Evaluation set is empty.')
    # Average annotators per image, then physical images; do not overweight triply labeled images.
    grouped = defaultdict(list)
    for row in rows: grouped[row['filename']].append(row)
    metrics = {key: float(np.mean([np.mean([r[key] for r in group]) for group in grouped.values()]))
               for key in ['pq', 'dice', 'iou', 'tp', 'fp', 'fn', 'split_relations', 'merge_relations']}
    return metrics, rows


In [ ]:
if RUN_CALIBRATION:
    selected_names = list(split_names['tune'])
    random.Random(CFG['seed']).shuffle(selected_names)
    if CFG['tune_image_limit'] is not None:
        selected_names = selected_names[:CFG['tune_image_limit']]
    if not selected_names:
        raise ValueError('Calibration requires at least one tuning image.')
    probability_cache = {n: predict_probability(train_lookup[n]) for n in tqdm(selected_names, desc='Tuning probabilities')}
    tuning_results = []
    for method in ['unet', 'contrast']:
        thresholds = CFG['thresholds'] if method == 'unet' else CFG['contrast_thresholds']
        for threshold, area in tqdm(list(itertools.product(thresholds, CFG['min_areas'])), desc=method):
            parameters = dict(method=method, min_area=area)
            parameters['threshold' if method == 'unet' else 'contrast_threshold'] = threshold
            scores, _ = evaluate(selected_names, parameters, probability_cache)
            tuning_results.append(dict(parameters=parameters, scores=scores))
    selected = {method: max((r for r in tuning_results if r['parameters']['method'] == method),
                           key=lambda r: r['scores']['pq'])['parameters'] for method in ['unet', 'contrast']}
    calibration = dict(config=dict(CFG), annotation_sha256=annotation_sha256,
        checkpoint_sha256=CHECKPOINT_SHA256,
        smoke_test=SMOKE_TEST, split_manifest=manifest, tuning_filenames=selected_names,
        selected=selected, candidates=tuning_results)
    write_json(OUTPUT_DIR / 'calibration.json', calibration)
    del probability_cache
else:
    with (INPUT_DIR / 'calibration.json').open() as stream:
        calibration = json.load(stream)
    if (calibration['config'] != CFG or calibration['annotation_sha256'] != annotation_sha256
            or calibration['smoke_test'] != SMOKE_TEST or calibration['split_manifest'] != manifest
            or calibration['checkpoint_sha256'] != CHECKPOINT_SHA256):
        raise ValueError('Saved calibration does not match this experiment.')
    selected = calibration['selected']
print('Selected on tuning PQ only:', selected)


## Optional controlled comparison: fresh 30 epochs versus fine-tune 10

**For your completed baseline:** copy this section's cells after calibration and run them manually. Do not rerun configuration or baseline training. The cells read the existing saved best weights, train in new subdirectories, then restore the baseline model and random-generator states. Set `RUN_EXTRA_EXPERIMENTS=True` only when ready; the default is off so Run All does not unexpectedly schedule another 40 epochs.

- `fresh_30`: initialize a new U-Net/ResNet18 from ImageNet, train for at most 30 epochs at 0.0003.
- `finetune_10`: initialize from your saved best baseline checkpoint, use a **new AdamW optimizer**, and train for at most 10 additional epochs at 0.0001. The lower rate is a conservative starting choice, not a proven best setting. This is not exact optimizer-state resume, and the baseline's best epoch may be earlier than 20.
- Both stop after five consecutive epochs without a tuning-loss improvement greater than 0.0001. Every strict best-loss checkpoint is saved, including smaller improvements; the patience counter uses the stated minimum improvement. These parameters are explicit experimental choices, not universal CV defaults.
- Both use the same day-grouped split, grayscale preprocessing, augmentation and Dice+BCE loss. The fine-tuning candidate includes the initial baseline as epoch zero, so it can retain those weights if additional training does not improve tuning loss.
- After training, both use the same threshold/area grid and same bounded tuning subset as the baseline. Compare tuning PQ, convergence, and runtime. **Do not choose between them using holdout/test scores.** These are two controlled candidates, not an exhaustive hyperparameter search or cross-validation study. Baseline tuning scores are already adaptively selected and are not unbiased generalization estimates.

The saved baseline configuration remains `architecture='Unet'`, `encoder='resnet18'`. FPN is supported but not selected. Both have hidden feature layers. ResNet18 is not a three-layer CNN; it is a deep residual encoder plus a decoder. `activation=None` applies only to the output head: hidden ReLU activations remain, BCE consumes logits, and inference explicitly applies sigmoid.

Documentation basis: scikit-learn [parameter search](https://scikit-learn.org/stable/modules/grid_search.html) recommends budgeted randomized search and log-scale continuous learning-rate distributions when searching wider spaces; `RandomizedSearchCV` does not directly train this PyTorch model without an estimator adapter. Ultralytics [training](https://docs.ultralytics.com/modes/train/) and [tuning](https://docs.ultralytics.com/guides/hyperparameter-tuning/) document epochs, patience, image size, batch size and learning-rate tuning. YOLO defaults are not validated U-Net/solar-filament optima. Changing resolution or batch size is deliberately deferred to a separate controlled search rather than mixing it into this continuation experiment.

### Rendered mathematics (Kaggle Markdown/MathJax)

$$p_i=\sigma(z_i)=\frac{1}{1+e^{-z_i}}$$

$$L_{\mathrm{BCE}}=-\frac{1}{N}\sum_i\left[y_i\log p_i+(1-y_i)\log(1-p_i)\right]$$

$$L_{\mathrm{Dice}}=1-\frac{1}{B}\sum_{b=1}^{B}\frac{2\sum_i p_{bi}y_{bi}+1}{\sum_i p_{bi}+\sum_i y_{bi}+1},\qquad L=\tfrac12 L_{\mathrm{BCE}}+\tfrac12 L_{\mathrm{Dice}}$$

$$\mathrm{IoU}=\frac{TP_{\mathrm{pixel}}}{TP_{\mathrm{pixel}}+FP_{\mathrm{pixel}}+FN_{\mathrm{pixel}}},\qquad F1_{\mathrm{pixel}}=\mathrm{Dice}=\frac{2TP_{\mathrm{pixel}}}{2TP_{\mathrm{pixel}}+FP_{\mathrm{pixel}}+FN_{\mathrm{pixel}}}$$

$$\mathrm{PQ}=\frac{\sum_{(p,g)\in\mathrm{matches}}\mathrm{IoU}(p,g)}{TP_{\mathrm{object}}+\tfrac12 FP_{\mathrm{object}}+\tfrac12 FN_{\mathrm{object}}}$$

These are segmentation/classification losses and metrics, not continuous-target regression models. The leaderboard value you reported (0.7) cannot establish your model's score or a guaranteed target without a comparable evaluation.

In [ ]:
RUN_EXTRA_EXPERIMENTS = False  # Set True to run both candidates; baseline is not overwritten.
EXTRA_RUN_TAG = 'convergence_v1'  # Change for a new comparison; existing trial files are protected.
EXPERIMENT_PATIENCE = 5
EXPERIMENT_MIN_DELTA = 1e-4
EXTRA_CANDIDATES = [
    dict(name='fresh_30', epochs=30, learning_rate=3e-4, initialize_from_baseline=False),
    dict(name='finetune_10', epochs=10, learning_rate=1e-4, initialize_from_baseline=True),
]


In [ ]:
class LossEarlyStopping:
    def __init__(self, patience, min_delta):
        if not isinstance(patience, int) or patience < 1 or not np.isfinite(min_delta) or min_delta < 0:
            raise ValueError('Patience must be positive and min_delta finite/nonnegative.')
        self.patience, self.min_delta = patience, min_delta
        self.reference_loss, self.bad_epochs = float('inf'), 0

    def update(self, loss):
        if not np.isfinite(loss):
            raise FloatingPointError('Early stopping received a nonfinite loss.')
        if loss < self.reference_loss - self.min_delta:
            self.reference_loss, self.bad_epochs = loss, 0
        else:
            self.bad_epochs += 1
        return self.bad_epochs >= self.patience

def run_candidate_training(candidate, baseline_checkpoint, trial_dir):
    trial_config = dict(CFG, epochs=candidate['epochs'], learning_rate=candidate['learning_rate'])
    if trial_config['epochs'] < 1 or not np.isfinite(trial_config['learning_rate']) or trial_config['learning_rate'] <= 0:
        raise ValueError('Candidate epochs and learning rate must be positive.')
    trial_dir.mkdir(parents=True, exist_ok=False)
    random.seed(CFG['seed']); np.random.seed(CFG['seed']); torch.manual_seed(CFG['seed'])
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(CFG['seed'])
    trial_model = build_model(trial_config, pretrained=not candidate['initialize_from_baseline'] and CFG['pretrained']).to(DEVICE)
    if candidate['initialize_from_baseline']:
        trial_model.load_state_dict(baseline_checkpoint['state_dict'], strict=True)
    optimizer = torch.optim.AdamW(trial_model.parameters(), lr=trial_config['learning_rate'])
    training_loader = DataLoader(FilamentDataset(split_records['train'], augment=True),
        batch_size=CFG['batch_size'], shuffle=True, num_workers=CFG['workers'],
        drop_last=len(split_records['train']) % CFG['batch_size'] == 1)
    validation_loader = DataLoader(FilamentDataset(split_records['tune']),
        batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['workers'])
    if not len(training_loader) or not len(validation_loader):
        raise ValueError('Candidate requires nonempty training and tuning loaders.')
    stopper = LossEarlyStopping(EXPERIMENT_PATIENCE, EXPERIMENT_MIN_DELTA)
    best_loss, best_epoch, trial_history = float('inf'), 0, []
    trial_checkpoint_path = trial_dir / 'best_model.pt'
    started = time.perf_counter()

    def save_candidate(epoch, loss):
        torch.save(dict(state_dict={k: v.detach().cpu() for k, v in trial_model.state_dict().items()},
            config=trial_config, epoch=epoch, tune_loss=loss, annotation_sha256=annotation_sha256,
            smoke_test=SMOKE_TEST, initialization=candidate,
            baseline_checkpoint_sha256=CHECKPOINT_SHA256,
            baseline_best_epoch=baseline_checkpoint['epoch'],
            early_stopping=dict(patience=EXPERIMENT_PATIENCE, min_delta=EXPERIMENT_MIN_DELTA)), trial_checkpoint_path)

    if candidate['initialize_from_baseline']:
        trial_model.eval()
        total, count = 0.0, 0
        with torch.inference_mode():
            for images, targets in validation_loader:
                loss = segmentation_loss(trial_model(images.to(DEVICE)), targets.to(DEVICE))
                if not torch.isfinite(loss): raise FloatingPointError('Nonfinite initial tuning loss.')
                total += loss.item()*len(images); count += len(images)
        best_loss = total/count
        stopper.update(best_loss)
        save_candidate(0, best_loss)
        trial_history.append(dict(epoch=0, train_loss=None, tune_loss=best_loss))

    stopped_early = False
    for epoch in range(1, trial_config['epochs']+1):
        totals = {}
        for phase, loader in [('train', training_loader), ('tune', validation_loader)]:
            trial_model.train(phase == 'train')
            total, count = 0.0, 0
            with torch.set_grad_enabled(phase == 'train'):
                for images, targets in tqdm(loader, desc=f'{candidate["name"]} {epoch} {phase}'):
                    images, targets = images.to(DEVICE), targets.to(DEVICE)
                    if phase == 'train': optimizer.zero_grad(set_to_none=True)
                    loss = segmentation_loss(trial_model(images), targets)
                    if not torch.isfinite(loss): raise FloatingPointError('Nonfinite candidate loss.')
                    if phase == 'train':
                        loss.backward(); optimizer.step()
                    total += loss.item()*len(images); count += len(images)
            totals[phase] = total/count
        trial_history.append(dict(epoch=epoch, train_loss=totals['train'], tune_loss=totals['tune']))
        write_json(trial_dir / 'training_history.json', trial_history)
        if totals['tune'] < best_loss:
            best_loss, best_epoch = totals['tune'], epoch
            save_candidate(epoch, best_loss)
        stopped_early = stopper.update(totals['tune'])
        print(candidate['name'], trial_history[-1], 'patience used:', stopper.bad_epochs)
        if stopped_early:
            print('Early stopping:', candidate['name'], 'after epoch', epoch)
            break
    summary = dict(candidate=candidate, config=trial_config, best_epoch=best_epoch, best_tune_loss=best_loss,
        epochs_executed=epoch, stopped_early=stopped_early, seconds=time.perf_counter()-started,
        baseline_best_epoch=baseline_checkpoint['epoch'], smoke_test=SMOKE_TEST,
        device=str(DEVICE), checkpoint=str(trial_checkpoint_path),
        early_stopping=dict(patience=EXPERIMENT_PATIENCE, min_delta=EXPERIMENT_MIN_DELTA))
    write_json(trial_dir / 'training_summary.json', summary)
    write_json(trial_dir / 'split_manifest.json', manifest)
    del trial_model, optimizer
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return summary


In [ ]:
if not RUN_EXTRA_EXPERIMENTS:
    print('Extra training is off. Set RUN_EXTRA_EXPERIMENTS=True in the comparison settings to run both candidates.')
else:
    if not EXTRA_RUN_TAG or Path(EXTRA_RUN_TAG).name != EXTRA_RUN_TAG or EXTRA_RUN_TAG in {'.', '..'}:
        raise ValueError('EXTRA_RUN_TAG must be a nonempty directory name, not a path.')
    names = [c['name'] for c in EXTRA_CANDIDATES]
    if not names or len(names) != len(set(names)) or any(Path(n).name != n or n in {'.', '..'} for n in names):
        raise ValueError('Candidates need unique, safe directory names.')
    comparison_dir = OUTPUT_DIR / EXTRA_RUN_TAG
    if comparison_dir.exists():
        raise FileExistsError('Comparison directory exists; choose a new EXTRA_RUN_TAG to preserve results.')
    baseline_saved = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
    if (baseline_saved['config'] != CFG or baseline_saved['annotation_sha256'] != annotation_sha256
            or baseline_saved['smoke_test'] != SMOKE_TEST
            or hashlib.sha256(CHECKPOINT_PATH.read_bytes()).hexdigest() != CHECKPOINT_SHA256):
        raise ValueError('Baseline checkpoint/configuration changed; reload the original experiment first.')
    if calibration['checkpoint_sha256'] != CHECKPOINT_SHA256:
        raise ValueError('Baseline calibration does not match the saved weights.')
    comparison_dir.mkdir()
    baseline_model, baseline_training_mode = model, model.training
    random_state, numpy_state, torch_state = random.getstate(), np.random.get_state(), torch.get_rng_state()
    cuda_states = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    comparison_results = []
    comparison_paths = [comparison_dir / c['name'] for c in EXTRA_CANDIDATES]
    try:
        for candidate, trial_dir in zip(EXTRA_CANDIDATES, comparison_paths):
            summary = run_candidate_training(candidate, baseline_saved, trial_dir)
            saved_trial = torch.load(trial_dir / 'best_model.pt', map_location='cpu', weights_only=True)
            model = build_model(saved_trial['config'], pretrained=False).to(DEVICE)
            model.load_state_dict(saved_trial['state_dict'], strict=True); model.eval()
            tuning_names = list(calibration['tuning_filenames'])
            probability_cache = {n: predict_probability(train_lookup[n]) for n in tqdm(tuning_names, desc='Candidate tuning predictions')}
            candidate_scores = []
            for threshold, area in itertools.product(CFG['thresholds'], CFG['min_areas']):
                parameters = dict(method='unet', threshold=threshold, min_area=area)
                scores, _ = evaluate(tuning_names, parameters, probability_cache)
                candidate_scores.append(dict(parameters=parameters, scores=scores))
            if not candidate_scores: raise ValueError('Calibration grid is empty.')
            winner = max(candidate_scores, key=lambda result: result['scores']['pq'])
            trial_calibration = dict(config=saved_trial['config'], annotation_sha256=annotation_sha256,
                checkpoint_sha256=hashlib.sha256((trial_dir / 'best_model.pt').read_bytes()).hexdigest(),
                smoke_test=SMOKE_TEST, split_manifest=manifest, tuning_filenames=tuning_names,
                selected=dict(unet=winner['parameters'], contrast=selected['contrast']),
                candidates=candidate_scores)
            write_json(trial_dir / 'calibration.json', trial_calibration)
            comparison_results.append(dict(**summary, tuning_scores=winner['scores'], parameters=winner['parameters']))
            write_json(comparison_dir / 'comparison_results.json', comparison_results)
            del probability_cache, saved_trial
            model = baseline_model
            if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    finally:
        model = baseline_model; model.train(baseline_training_mode)
        random.setstate(random_state); np.random.set_state(numpy_state); torch.set_rng_state(torch_state)
        if cuda_states is not None: torch.cuda.set_rng_state_all(cuda_states)
    print(json.dumps(comparison_results, indent=2))
    print('Baseline restored. Comparison used tuning data only; no holdout/test evaluation or submission occurred.')


In [ ]:
if RUN_EXTRA_EXPERIMENTS:
    curve_figure, curve_axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
    for candidate, trial_dir in zip(EXTRA_CANDIDATES, comparison_paths):
        trial_history = json.loads((trial_dir / 'training_history.json').read_text())
        curve_axes[0].plot([r['epoch'] for r in trial_history if r['train_loss'] is not None],
            [r['train_loss'] for r in trial_history if r['train_loss'] is not None], label=candidate['name'])
        curve_axes[1].plot([r['epoch'] for r in trial_history], [r['tune_loss'] for r in trial_history], label=candidate['name'])
    for axis, title in zip(curve_axes, ['Training loss', 'Tuning loss (early stopping)']):
        axis.set_title(title); axis.set_xlabel('Epochs within this candidate'); axis.set_ylabel('Dice+BCE'); axis.legend(); axis.grid(alpha=0.3)
    curve_figure.savefig(comparison_dir / 'convergence_comparison.png', dpi=150)
    plt.show()
    print('Fine-tuning epoch zero is the baseline best checkpoint, not a newly initialized model.')


### Explicitly select a candidate for downstream inference (optional)

The comparison restores the baseline by default. To export/visualize one candidate instead, set `USE_EXTRA_CANDIDATE` below to its name **after** reviewing tuning results, then run this cell followed by the existing holdout/visualization/export/evidence cells. It updates weights, config, calibration, checkpoint identity, and output directory together. Do not rerun baseline training or baseline calibration after selection. The original weights remain in their original directory. Existing candidate outputs are kept beside its own checkpoint.

If you already examined the baseline holdout, do not claim a new untouched model-selection benchmark from repeatedly consulting it. Choose candidates on tuning data; document any repeated holdout inspection. To do another comparison after selecting a candidate, first restore the original baseline session/configuration.

In [ ]:
USE_EXTRA_CANDIDATE = None  # 'fresh_30' or 'finetune_10'; None keeps the currently loaded model.
if USE_EXTRA_CANDIDATE is not None:
    if not RUN_EXTRA_EXPERIMENTS or USE_EXTRA_CANDIDATE not in [c['name'] for c in EXTRA_CANDIDATES]:
        raise ValueError('Run the comparison first and select a configured candidate.')
    chosen_dir = comparison_dir / USE_EXTRA_CANDIDATE
    chosen_checkpoint_path = chosen_dir / 'best_model.pt'
    chosen_checkpoint = torch.load(chosen_checkpoint_path, map_location='cpu', weights_only=True)
    chosen_calibration = json.loads((chosen_dir / 'calibration.json').read_text())
    chosen_hash = hashlib.sha256(chosen_checkpoint_path.read_bytes()).hexdigest()
    if (chosen_calibration['checkpoint_sha256'] != chosen_hash
            or chosen_calibration['config'] != chosen_checkpoint['config']
            or chosen_calibration['split_manifest'] != manifest
            or chosen_checkpoint['annotation_sha256'] != annotation_sha256
            or chosen_checkpoint['smoke_test'] != SMOKE_TEST):
        raise ValueError('Candidate weights/calibration do not match this experiment.')
    chosen_model = build_model(chosen_checkpoint['config'], pretrained=False).to(DEVICE)
    chosen_model.load_state_dict(chosen_checkpoint['state_dict'], strict=True); chosen_model.eval()
    model, checkpoint, calibration = chosen_model, chosen_checkpoint, chosen_calibration
    CFG = dict(checkpoint['config'])
    selected = calibration['selected']
    CHECKPOINT_PATH, CHECKPOINT_SHA256 = chosen_checkpoint_path, chosen_hash
    OUTPUT_DIR = INPUT_DIR = chosen_dir
    RUN_NAME = chosen_dir.name
    RUN_TRAINING = RUN_CALIBRATION = False
    print('Active candidate:', RUN_NAME, '| best additional epoch:', checkpoint['epoch'], '| outputs:', OUTPUT_DIR)
else:
    print('No candidate switch requested; currently loaded model remains active.')


## 4. Compare learned and classical CV on an untouched holdout

The contrast baseline separates locally dark pixels from a smoothed background inside an eroded solar disk; it is an intensity-based **heuristic**, not a trained SVM or ESAIR Anomalyzer. It can confuse sunspots, disk/limb artifacts and clouds with filaments. Since observations are grayscale, RGB color clustering would add no independent color information. A learned model can exploit shape and context instead of intensity alone.

PQ uses one-to-one matching at IoU > 0.5, ignoring chirality. Split/merge diagnostics count overlap relations at IoU > 0.1 and are not an official metric. Each annotator is evaluated separately; we average annotators within a physical image, then images. Both-empty prediction/truth scores are defined as one; one-empty scores are zero. Keep holdout untouched after reading these results. Use separate grouped folds for further experiments.

In [ ]:
holdout_results = {}
holdout_rows = {}
for method in ['unet', 'contrast']:
    started = time.perf_counter()
    scores, rows = evaluate(tqdm(split_names['holdout'], desc=f'Holdout {method}'), selected[method])
    holdout_results[method] = dict(scores=scores, seconds=time.perf_counter()-started,
                                  image_count=len(split_names['holdout']), annotation_set_count=len(rows))
    holdout_rows[method] = rows
    with (OUTPUT_DIR / f'holdout_{method}.csv').open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader(); writer.writerows(rows)
write_json(OUTPUT_DIR / 'holdout_summary.json', holdout_results)
print(json.dumps(holdout_results, indent=2))
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for axis, metric in zip(axes, ['pq', 'dice', 'iou']):
    for method, rows in holdout_rows.items():
        by_file = defaultdict(list)
        for row in rows: by_file[row['filename']].append(row[metric])
        axis.hist([np.mean(v) for v in by_file.values()], bins=np.linspace(0, 1, 11), alpha=0.5, label=method)
    axis.set_title(f'Holdout {metric.upper()}'); axis.legend()
fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'holdout_distributions.png', dpi=150)
plt.show()

preview_name = split_names['holdout'][0]
predicted = instance_rles(predict_probability(train_lookup[preview_name]),
                          selected['unet']['threshold'], selected['unet']['min_area'])
record = records_by_name[preview_name][0]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
with Image.open(train_lookup[preview_name]) as image: full_image = np.asarray(image)
for axis in axes: axis.imshow(full_image, cmap='gray'); axis.axis('off')
axes[0].set_title('Grayscale observation')
axes[1].contour(union_mask(gt_rles(record['id'])), levels=[0.5], colors=['red'], linewidths=0.5)
axes[1].set_title('One annotator ground truth')
if predicted:
    axes[2].contour(union_mask(predicted), levels=[0.5], colors=['cyan'], linewidths=0.5)
axes[2].set_title('U-Net + components (no geometry grouping)')
fig.savefig(OUTPUT_DIR / 'holdout_overlay.png', dpi=150, bbox_inches='tight'); plt.show()


## Visual inspection: animated predictions and filament bounding boxes

**Already training on Kaggle? Let the current 20 epochs finish.** Add these cells after calibration, then run only the new cells; do not rerun configuration or training. They reuse the best saved checkpoint already loaded into `model`, not necessarily the epoch-20 weights. They require the prediction helpers and `selected` calibration parameters to have run. No optimizer updates or uploads occur here.

The animation applies those fixed weights to each selected image. It is not a visualization of changing weights or a continuous solar event: filenames span different dates and observing stations. By default it cycles through all 180 test images in filename order, with colored instance masks and boxes. Set `VIS_SPLIT='tune'` to inspect labeled calibration images when deciding changes. Do not select thresholds from holdout/test appearance and then describe the original holdout as unbiased.

`VIS_FRAME_LIMIT=None` renders the entire selected split. Use a small limit for a quick preview. Each frame runs inference and full-resolution component extraction; the final display is reduced to 512 pixels. Palette-compressed frames use roughly 45 MiB for 180 frames plus model/inference overhead. The GIF uses Pillow, so FFmpeg is not needed. Output files include the split name. Boxes are derived from predicted masks, not a separately trained detector; touching filaments can share a box and fragmented filaments can get multiple boxes.

In [ ]:
from PIL import ImageDraw
from IPython.display import Image as NotebookImage, display

VIS_SPLIT = 'test'  # 'tune' for labeled calibration-set inspection.
VIS_FRAME_LIMIT = None  # None includes every image in this split.
VIS_FRAME_DURATION_MS = 500
VIS_DISPLAY_SIZE = 512
if VIS_SPLIT not in {'test', 'tune'}:
    raise ValueError('Choose test for inspection or tune for calibration; do not tune on holdout.')
if VIS_FRAME_LIMIT is not None and (not isinstance(VIS_FRAME_LIMIT, int) or VIS_FRAME_LIMIT < 1):
    raise ValueError('VIS_FRAME_LIMIT must be a positive integer or None.')
if VIS_FRAME_DURATION_MS < 20 or VIS_DISPLAY_SIZE < 64:
    raise ValueError('Use duration >= 20 ms and display size >= 64 pixels.')
visualization_paths = (list(test_paths) if VIS_SPLIT == 'test'
                       else [train_lookup[name] for name in split_names['tune']])
if VIS_FRAME_LIMIT is not None:
    visualization_paths = visualization_paths[:VIS_FRAME_LIMIT]
if not visualization_paths:
    raise ValueError('No images available for visualization.')
visualization_parameters = dict(selected['unet'])
model.eval()

def visual_prediction(path):
    probability = predict_probability(path)
    rles = instance_rles(probability, visualization_parameters['threshold'],
                         visualization_parameters['min_area'])
    boxes = np.asarray(mask_utils.toBbox(rles), dtype=float).reshape(-1, 4) if rles else np.empty((0, 4))
    if not np.isfinite(boxes).all():
        raise ValueError(f'Nonfinite predicted box for {path.name}')
    return probability, rles, boxes

def prediction_frame(path, rles, boxes, frame_index, frame_count):
    with Image.open(path) as image:
        frame = image.resize((VIS_DISPLAY_SIZE, VIS_DISPLAY_SIZE), Image.Resampling.BILINEAR).convert('RGBA')
    scale = VIS_DISPLAY_SIZE / FULL_SHAPE[1]
    colors = [(0, 220, 255), (255, 170, 0), (100, 255, 100), (255, 100, 220), (180, 160, 255)]
    overlay = Image.new('RGBA', frame.size, (0, 0, 0, 0))
    for index, (rle, box) in enumerate(zip(rles, boxes), 1):
        color = colors[(index-1) % len(colors)]
        mask = Image.fromarray(mask_utils.decode(rle) * np.uint8(255)).resize(frame.size, Image.Resampling.NEAREST)
        opacity = mask.point(lambda value: value * 90 // 255)
        overlay.paste(Image.new('RGBA', frame.size, (*color, 255)), (0, 0), opacity)
    frame = Image.alpha_composite(frame, overlay)
    drawing = ImageDraw.Draw(frame)
    for index, (x, y, width, height) in enumerate(boxes, 1):
        color = colors[(index-1) % len(colors)]
        drawing.rectangle((x*scale, y*scale, (x+width)*scale, (y+height)*scale), outline=(*color, 255), width=2)
        drawing.text((x*scale, max(32, y*scale-12)), str(index), fill=(*color, 255))
    drawing.rectangle((0, 0, VIS_DISPLAY_SIZE, 31), fill=(0, 0, 0, 255))
    drawing.text((5, 3), f'{VIS_SPLIT} {frame_index}/{frame_count}: {path.name}', fill='white')
    drawing.text((5, 17), f'{len(rles)} instances | best epoch {checkpoint["epoch"]}', fill='white')
    return frame.convert('RGB').quantize(colors=256)

frames, animation_audit = [], []
for frame_index, path in enumerate(tqdm(visualization_paths, desc=f'Animate {VIS_SPLIT}'), 1):
    probability, rles, boxes = visual_prediction(path)
    frames.append(prediction_frame(path, rles, boxes, frame_index, len(visualization_paths)))
    animation_audit.append(dict(filename=path.name, instances=len(rles), boxes_xywh=boxes.tolist()))
animation_path = OUTPUT_DIR / f'{VIS_SPLIT}_predictions.gif'
temporary_animation_path = animation_path.with_suffix('.gif.tmp')
frames[0].save(temporary_animation_path, format='GIF', save_all=True, append_images=frames[1:],
               duration=VIS_FRAME_DURATION_MS, loop=0, disposal=2, optimize=False)
with Image.open(temporary_animation_path) as saved_animation:
    if saved_animation.size != (VIS_DISPLAY_SIZE, VIS_DISPLAY_SIZE) or saved_animation.n_frames != len(frames):
        raise ValueError('Saved animation dimensions/frame count failed validation.')
temporary_animation_path.replace(animation_path)
write_json(OUTPUT_DIR / f'{VIS_SPLIT}_animation_audit.json', dict(split=VIS_SPLIT,
    checkpoint_sha256=CHECKPOINT_SHA256, checkpoint_epoch=checkpoint['epoch'],
    parameters=visualization_parameters, frame_duration_ms=VIS_FRAME_DURATION_MS, images=animation_audit))
del frames
print('Animation:', animation_path, '| frames:', len(visualization_paths))
display(NotebookImage(filename=str(animation_path), width=VIS_DISPLAY_SIZE))


In [ ]:
from matplotlib.patches import Rectangle

# Zero-based indices into the split selected in the animation cell; edit to inspect other images.
BOX_IMAGE_INDICES = sorted({0, len(visualization_paths)//2, len(visualization_paths)-1})
if not BOX_IMAGE_INDICES or any(not isinstance(i, int) or i < 0 or i >= len(visualization_paths) for i in BOX_IMAGE_INDICES):
    raise ValueError('BOX_IMAGE_INDICES must reference images in visualization_paths.')
box_figure, box_axes = plt.subplots(len(BOX_IMAGE_INDICES), 2,
    figsize=(14, 6*len(BOX_IMAGE_INDICES)), squeeze=False, constrained_layout=True)
box_records = []
for row, image_index in enumerate(BOX_IMAGE_INDICES):
    path = visualization_paths[image_index]
    probability, rles, boxes = visual_prediction(path)
    with Image.open(path) as image: observation = np.asarray(image)
    box_axes[row, 0].imshow(full_probability(probability), cmap='magma', vmin=0, vmax=1)
    box_axes[row, 0].set_title(f'{path.name}: foreground probability')
    box_axes[row, 1].imshow(observation, cmap='gray', vmin=0, vmax=255)
    if rles:
        foreground = union_mask(rles)
        box_axes[row, 1].imshow(np.ma.masked_where(foreground == 0, foreground),
                               cmap='cool', vmin=0, vmax=1, alpha=0.3)
    for instance_index, (x, y, width, height) in enumerate(boxes, 1):
        color = plt.get_cmap('tab10')((instance_index-1) % 10)
        box_axes[row, 1].add_patch(Rectangle((x, y), width, height, fill=False, edgecolor=color, linewidth=1.5))
        box_axes[row, 1].text(x, max(0, y-8), str(instance_index), color=color, fontsize=9,
                              bbox=dict(facecolor='black', alpha=0.65, edgecolor='none', pad=1))
        box_records.append(dict(filename=path.name, instance=instance_index,
                                x=float(x), y=float(y), width=float(width), height=float(height)))
    box_axes[row, 1].set_title(f'{len(rles)} mask-derived boxes (original pixel coordinates)')
    for axis in box_axes[row]: axis.axis('off')
box_gallery_path = OUTPUT_DIR / f'{VIS_SPLIT}_filament_boxes.png'
box_figure.savefig(box_gallery_path, dpi=160, bbox_inches='tight')
with (OUTPUT_DIR / f'{VIS_SPLIT}_filament_boxes.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=['filename', 'instance', 'x', 'y', 'width', 'height'])
    writer.writeheader(); writer.writerows(box_records)
print('Bounding-box gallery:', box_gallery_path)
plt.show()


### MP4 videos: labeled observations and test predictions

Run the video cell after the animation cell (which defines `visual_prediction` and `prediction_frame`). Set `RUN_MP4_VIDEOS=True`. It streams frames into FFmpeg rather than retaining them all in memory. Left: observation; center: one human annotation set for labeled images, or probability heatmap for test; right: predicted instance overlays/boxes. Labeled frames identify train/tune/holdout membership. Do not select thresholds on training/holdout appearance. Fixed weights process independent observations, not temporal evolution or weight updates.

GPU inference uses the current model device; plotting, COCO masks and H.264 encoding run on CPU. CUDA works on Kaggle. Local Apple MPS can accelerate inference if the model and `DEVICE` are both moved to MPS; it is not NVIDIA CUDA. Real videos require the actual trained checkpoint and calibration, not just downloaded JPEGs. If FFmpeg is unavailable, install `imageio-ffmpeg` in the kernel first. No upload or training occurs.

GitHub preparation: download the MP4s and their audits after reviewing them. A public repository README can link to Release assets. Do not publish competition-derived images/videos before confirming allowed sharing and attribution. GitHub hosting is not configured by this notebook.

In [ ]:
RUN_MP4_VIDEOS = False  # Set True to create both videos with the currently loaded calibrated weights.
MP4_LIMIT = None  # None processes all images; use 10 for a preview.
MP4_FPS = 2

def render_review_videos(limit=None, fps=2):
    from matplotlib.animation import FFMpegWriter
    from PIL import ImageDraw
    from IPython.display import Video, display
    if limit is not None and (not isinstance(limit, int) or limit < 1):
        raise ValueError('MP4 limit must be positive or None.')
    if not np.isfinite(fps) or fps <= 0:
        raise ValueError('MP4 FPS must be finite and positive.')
    if visualization_parameters != selected['unet']:
        raise ValueError('Model selection changed: rerun the animation setup with the current parameters.')
    encoder = shutil.which('ffmpeg')
    if encoder is None:
        import importlib.util
        if importlib.util.find_spec('imageio_ffmpeg') is None:
            raise RuntimeError('Install imageio-ffmpeg in this kernel, then rerun.')
        import imageio_ffmpeg
        encoder = imageio_ffmpeg.get_ffmpeg_exe()
    membership = {name: split for split, names in split_names.items() for name in names}
    model.eval()
    output_videos = []
    with plt.rc_context({'animation.ffmpeg_path': encoder}):
        for split, paths in [('labeled', train_paths), ('test', test_paths)]:
            paths = sorted(paths)[:limit] if limit is not None else sorted(paths)
            if not paths: raise ValueError(f'No images for {split}.')
            video_path = OUTPUT_DIR / f'{split}_filament_review.mp4'
            temporary_path = video_path.with_suffix('.tmp.mp4')
            side = VIS_DISPLAY_SIZE
            figure, axis = plt.subplots(figsize=(12, 4), dpi=128)
            figure.subplots_adjust(left=0, right=1, top=1, bottom=0)
            axis.axis('off')
            artist = axis.imshow(np.zeros((side, side*3, 3), dtype=np.uint8))
            writer = FFMpegWriter(fps=fps, codec='libx264', extra_args=['-crf', '22', '-pix_fmt', 'yuv420p'])
            audit = []
            started = time.perf_counter()
            try:
                with torch.inference_mode(), writer.saving(figure, str(temporary_path), dpi=128):
                    for index, path in enumerate(tqdm(paths, desc=f'MP4 {split}'), 1):
                        probability, rles, boxes = visual_prediction(path)
                        with Image.open(path) as image:
                            observation = image.resize((side, side), Image.Resampling.BILINEAR).convert('RGB')
                        label_id = None
                        if split == 'labeled':
                            record = sorted(records_by_name[path.name], key=lambda r: str(r['id']))[0]
                            label_id = record['id']
                            truth = Image.fromarray(union_mask(gt_rles(label_id))*np.uint8(255)).resize((side, side), Image.Resampling.NEAREST)
                            middle = Image.blend(observation, Image.composite(Image.new('RGB', (side, side), 'lime'), observation, truth), 0.5)
                            middle_title = 'Human target: ' + str(label_id)
                        else:
                            display_probability = np.asarray(Image.fromarray(probability).resize((side, side), Image.Resampling.BILINEAR))
                            middle = Image.fromarray((plt.get_cmap('magma')(display_probability)[:, :, :3]*255).astype(np.uint8))
                            middle_title = 'Probability (0 to 1)'
                        right = prediction_frame(path, rles, boxes, index, len(paths)).convert('RGB')
                        canvas = Image.new('RGB', (side*3, side))
                        canvas.paste(observation, (0, 0)); canvas.paste(middle, (side, 0)); canvas.paste(right, (side*2, 0))
                        drawing = ImageDraw.Draw(canvas)
                        for x, title in [(0, f'{path.name} | {membership.get(path.name, "test")}'), (side, middle_title)]:
                            drawing.rectangle((x, 0, x+side, 31), fill='black')
                            drawing.text((x+4, 8), title, fill='white')
                        artist.set_data(np.asarray(canvas)); writer.grab_frame()
                        audit.append(dict(filename=path.name, partition=membership.get(path.name, 'test'),
                                          annotation_id=label_id, instances=len(rles), boxes_xywh=boxes.tolist()))
            finally:
                plt.close(figure)
            if not temporary_path.is_file() or temporary_path.stat().st_size == 0:
                raise RuntimeError('Video encoding produced no output.')
            temporary_path.replace(video_path)
            write_json(OUTPUT_DIR / f'{split}_video_audit.json', dict(checkpoint_sha256=CHECKPOINT_SHA256,
                checkpoint_epoch=checkpoint['epoch'], parameters=visualization_parameters,
                fps=fps, frame_count=len(audit), render_seconds=time.perf_counter()-started, images=audit))
            print('Saved:', video_path)
            display(Video(filename=str(video_path), embed=True, width=1100))
            output_videos.append(video_path)
    return output_videos

if RUN_MP4_VIDEOS:
    video_paths = render_review_videos(MP4_LIMIT, MP4_FPS)
else:
    print('Video rendering is off. Set RUN_MP4_VIDEOS=True when calibrated weights are loaded.')


## 5. Test inference and strict submission validation

The U-Net checkpoint is the predeclared submission model, regardless of which method wins on holdout. Probabilities are bilinearly restored to 2048 x 2048 **before** thresholding and extracting instances. No fabricated foreground or empty RLE row is inserted. If some images have zero detections, export writes submission_candidate.csv and a separate candidate manifest, explicitly omitting filament rows for those images. Kaggle acceptance of that representation is unverified, so automatic upload remains blocked. An entirely empty prediction set still raises an error. Do not tune against hidden test labels or invent detections to satisfy a file checker.

All retained filament masks are decoded and round-tripped before writing. Files are written atomically after verifying complete test inference, audited instance counts and unique IDs. The first three test images, probability maps, and mask outlines are saved in `test_preview.png`. `SMOKE_TEST=True` uses a separately named CSV which the upload cell refuses. Outputs are under `/kaggle/working/<RUN_NAME>/`.

For offline reproduction, attach the experiment folder containing `best_model.pt`, `calibration.json`, and `split_manifest.json`, restore the same configuration and disable training/calibration. This export path has no pretrained-weight download.

In [ ]:
submission_rows, prediction_audit = [], []
submission_path = None
empty_predictions = []
started = time.perf_counter()
preview_count = min(3, len(test_paths))
preview_figure, preview_axes = plt.subplots(preview_count, 3, figsize=(12, 4*preview_count), squeeze=False)
for test_index, path in enumerate(tqdm(test_paths, desc='Full-resolution test instances')):
    probability = predict_probability(path)
    predicted = instance_rles(probability, selected['unet']['threshold'], selected['unet']['min_area'])
    if test_index < preview_count:
        with Image.open(path) as image: observation = np.asarray(image)
        preview_axes[test_index, 0].imshow(observation, cmap='gray')
        preview_axes[test_index, 0].set_title(path.name, fontsize=9)
        preview_axes[test_index, 1].imshow(full_probability(probability), cmap='magma', vmin=0, vmax=1)
        preview_axes[test_index, 1].set_title('Filament probability')
        preview_axes[test_index, 2].imshow(observation, cmap='gray')
        if predicted:
            preview_axes[test_index, 2].contour(union_mask(predicted), levels=[0.5], colors=['cyan'], linewidths=0.5)
        preview_axes[test_index, 2].set_title(f'{len(predicted)} predicted instances')
        for axis in preview_axes[test_index]: axis.axis('off')
    if not predicted: empty_predictions.append(path.name)
    prediction_audit.append(dict(filename=path.name, instances=len(predicted),
                                 foreground_pixels=int(mask_utils.area(mask_utils.merge(predicted))) if predicted else 0))
    for index, rle in enumerate(predicted, 1):
        counts = rle['counts'].decode('ascii')
        mask = mask_utils.decode(dict(size=list(FULL_SHAPE), counts=counts.encode('ascii')))
        if mask.shape != FULL_SHAPE or mask.sum() == 0:
            raise ValueError(f'Invalid/empty mask for {path.name}')
        roundtrip = mask_utils.encode(np.asfortranarray(mask, dtype=np.uint8))['counts'].decode('ascii')
        if roundtrip != counts:
            raise ValueError('COCO RLE roundtrip failed.')
        submission_rows.append(dict(filament_id=f'{path.stem}_{index}', segmentation_rle=counts))
preview_figure.tight_layout()
preview_figure.savefig(OUTPUT_DIR / 'test_preview.png', dpi=150, bbox_inches='tight'); plt.show()
write_json(OUTPUT_DIR / 'test_prediction_audit.json', dict(images=prediction_audit,
    seconds=time.perf_counter()-started, empty_predictions=empty_predictions, smoke_test=SMOKE_TEST))
processed_names = [record['filename'] for record in prediction_audit]
if len(processed_names) != len(test_paths) or set(processed_names) != {p.name for p in test_paths}:
    raise ValueError('Test inference is incomplete or contains duplicate images.')
if not submission_rows:
    raise ValueError('No filament predictions exist anywhere in the test set.')
ids = [row['filament_id'] for row in submission_rows]
if len(ids) != len(set(ids)):
    raise ValueError('Duplicate filament IDs.')
expected_counts = {Path(record['filename']).stem: record['instances'] for record in prediction_audit}
actual_counts = Counter(identifier.rsplit('_', 1)[0] for identifier in ids)
if dict(actual_counts) != {stem: count for stem, count in expected_counts.items() if count > 0}:
    raise ValueError('CSV instance counts do not match the test inference audit.')
expected_stems = {stem for stem, count in expected_counts.items() if count > 0}
if {identifier.rsplit('_', 1)[0] for identifier in ids} != expected_stems:
    raise ValueError('Submission image coverage differs from test files.')
filename = ('smoke_submission.csv' if SMOKE_TEST else
            'submission_candidate.csv' if empty_predictions else 'submission.csv')
submission_path = OUTPUT_DIR / filename
temporary_path = submission_path.with_suffix('.csv.tmp')
with temporary_path.open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=['filament_id', 'segmentation_rle'])
    writer.writeheader(); writer.writerows(submission_rows)
with temporary_path.open(newline='') as stream:
    reader = csv.DictReader(stream)
    if reader.fieldnames != ['filament_id', 'segmentation_rle'] or list(reader) != submission_rows:
        raise ValueError('CSV validation failed.')
temporary_path.replace(submission_path)
manifest_name = 'submission_candidate_manifest.json' if empty_predictions else 'submission_manifest.json'
write_json(OUTPUT_DIR / manifest_name, dict(smoke_test=SMOKE_TEST,
    zero_detection_images=empty_predictions,
    zero_detection_policy='unverified; omit filament rows' if empty_predictions else 'not applicable',
    rows=len(submission_rows), images=len(test_paths),
    csv_sha256=hashlib.sha256(submission_path.read_bytes()).hexdigest(),
    checkpoint_sha256=CHECKPOINT_SHA256, parameters=selected['unet']))
print('Locally validated CSV:', submission_path, 'rows:', len(submission_rows), 'processed images:', len(test_paths))
if empty_predictions:
    print('WARNING: candidate only; Kaggle acceptance of omitted zero-detection images is unverified.')
    print('Automatic upload is blocked for this candidate. Zero-detection images:', empty_predictions)


### If export stopped because some test images have no predictions

This is an export safeguard, not a training exception. The trained checkpoint, calibration and holdout results remain available. **Do not rerun training.** Run the diagnostic cell below manually after the export exception, and run the final evidence-saving cell manually if Run All stopped before it. Save/download the completed run's outputs before ending the Kaggle session.

The diagnostic distinguishes (a) no probability reaching the calibrated threshold from (b) foreground components all removed by the calibrated minimum-area rule. It shows the observation, probability heatmap and thresholded foreground before area filtering. It never changes thresholds, invents masks, or uploads a submission. A zero-detection image is not proven to contain no true filaments. Confirm the organizer's zero-detection representation policy before changing CSV coverage rules; do not lower thresholds on individual test images to force detections.

In [ ]:
if 'empty_predictions' not in globals():
    raise RuntimeError('Run test inference/export first; then run this cell manually if export stops.')
empty_diagnostics = []
if not empty_predictions:
    print('No zero-detection test images were reported by the latest export attempt.')
else:
    test_lookup = {path.name: path for path in test_paths}
    diagnostic_threshold = selected['unet']['threshold']
    diagnostic_min_area = selected['unet']['min_area']
    for filename in empty_predictions:
        if filename not in test_lookup:
            raise ValueError(f'Zero-detection filename is not in this test set: {filename}')
        probability = full_probability(predict_probability(test_lookup[filename]))
        foreground = probability >= diagnostic_threshold
        labels, component_count = ndi.label(foreground, structure=np.ones((3, 3), dtype=np.uint8))
        component_areas = np.bincount(labels.ravel())[1:]
        retained_count = int(np.count_nonzero(component_areas >= diagnostic_min_area))
        if retained_count:
            raise ValueError(f'{filename} now has detections: weights/parameters may have changed since export.')
        diagnostic = dict(filename=filename, threshold=diagnostic_threshold, min_area=diagnostic_min_area,
            probability_max=float(probability.max()), probability_p99=float(np.quantile(probability, 0.99)),
            pixels_above_threshold=int(foreground.sum()), components_before_area_filter=int(component_count),
            largest_component_area=int(component_areas.max()) if component_areas.size else 0,
            cause='below_probability_threshold' if not foreground.any() else 'all_components_below_minimum_area')
        empty_diagnostics.append(diagnostic)
        print(json.dumps(diagnostic, indent=2))
        diagnostic_figure, diagnostic_axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
        with Image.open(test_lookup[filename]) as image: observation = np.asarray(image)
        diagnostic_axes[0].imshow(observation, cmap='gray', vmin=0, vmax=255)
        diagnostic_axes[0].set_title(filename)
        heatmap = diagnostic_axes[1].imshow(probability, cmap='magma', vmin=0, vmax=1)
        diagnostic_axes[1].set_title(f'Probability; max={probability.max():.4f}')
        diagnostic_figure.colorbar(heatmap, ax=diagnostic_axes[1], fraction=0.046)
        diagnostic_axes[2].imshow(observation, cmap='gray', vmin=0, vmax=255)
        diagnostic_axes[2].imshow(np.ma.masked_where(~foreground, foreground), cmap='cool', vmin=0, vmax=1, alpha=0.6)
        diagnostic_axes[2].set_title(f'Before area filter: {int(foreground.sum())} pixels')
        for axis in diagnostic_axes: axis.axis('off')
        diagnostic_figure.savefig(OUTPUT_DIR / f'empty_diagnostic_{test_lookup[filename].stem}.png', dpi=150)
        plt.show()
    write_json(OUTPUT_DIR / 'empty_prediction_diagnostics.json', dict(
        checkpoint_sha256=CHECKPOINT_SHA256, parameters=selected['unet'], images=empty_diagnostics))


## Prepare the manual Kaggle submission

Run All validates the exported CSV and prepares `submission.csv` in `/kaggle/working`. This cell never calls the Kaggle API, even if an earlier setting enabled it. After saving the completed notebook version, use Kaggle's Submit button and select that output CSV. Blocking checks print `NOT READY` without raising; never submit a stale file. Zero-detection candidates retain their original masks and print an acceptance warning; local validation does not guarantee Kaggle acceptance.

In [ ]:
import csv
import hashlib
import json
import shutil
from pathlib import Path
from IPython.display import FileLink, display

# Manual workflow: Run All prepares the file; Kaggle's Submit button uploads it.
SUBMIT_TO_KAGGLE = False
submission_ready = False
manual_submission_path = None

def prepare_manual_submission():
    if SMOKE_TEST or checkpoint.get("smoke_test", True):
        print("BLOCKED: smoke-test predictions cannot be submitted.")
        return None
    if submission_path is None:
        print("BLOCKED: no CSV was produced. Inspect the test prediction audit.")
        return None
    source = Path(submission_path)
    if source.name not in {"submission.csv", "submission_candidate.csv"}:
        print("BLOCKED: unexpected submission filename:", source.name)
        return None
    manifest_name = (
        "submission_candidate_manifest.json"
        if source.name == "submission_candidate.csv"
        else "submission_manifest.json"
    )
    try:
        saved = json.loads((source.parent / manifest_name).read_text())
        with source.open(newline="") as stream:
            reader = csv.DictReader(stream)
            columns, rows = reader.fieldnames, list(reader)
        valid = (
            isinstance(saved, dict)
            and saved.get("smoke_test") is False
            and saved.get("parameters") == selected["unet"]
            and saved.get("images") == len(test_paths)
            and saved.get("checkpoint_sha256") == hashlib.sha256(CHECKPOINT_PATH.read_bytes()).hexdigest()
            and saved.get("csv_sha256") == hashlib.sha256(source.read_bytes()).hexdigest()
            and columns == ["filament_id", "segmentation_rle"]
            and bool(rows)
            and rows == submission_rows
            and saved.get("rows") == len(rows)
            and len({row["filament_id"] for row in rows}) == len(rows)
        )
        if not valid:
            print("BLOCKED: CSV, manifest, weights, or configuration failed validation. Rerun export.")
            return None
        destination = (
            Path("/kaggle/working/submission.csv")
            if Path("/kaggle/working").is_dir()
            else OUTPUT_DIR / "submission.csv"
        )
        if source.resolve() != destination.resolve():
            temporary = destination.with_suffix(".csv.tmp")
            shutil.copyfile(source, temporary)
            if hashlib.sha256(temporary.read_bytes()).hexdigest() != saved["csv_sha256"]:
                print("BLOCKED: copied CSV checksum differs; do not submit it.")
                temporary.unlink()
                return None
            temporary.replace(destination)
        if source.name == "submission_candidate.csv" or saved.get("zero_detection_images"):
            print("WARNING: some images have zero detections and no filament rows.")
            print("Kaggle acceptance of this representation is unverified; no masks were fabricated.")
        print("LOCAL CHECKS PASSED:", destination, "| filament rows:", len(rows))
        print("No API upload performed. Save Version / Run All, then select this CSV using Kaggle's Submit button.")
        return destination
    except (OSError, UnicodeError, json.JSONDecodeError, csv.Error) as error:
        print(f"BLOCKED: unable to prepare submission ({type(error).__name__}): {error}")
        return None

manual_submission_path = prepare_manual_submission()
submission_ready = manual_submission_path is not None
if submission_ready:
    display(FileLink(str(manual_submission_path)))
else:
    print("NOT READY: do not select an older CSV left in the output directory.")


## 7. Paper evidence and reproducibility package

After the full Kaggle run, use the saved metrics and figures in the organizer's four-page template. Do not equate a smoke test with research results or fabricate leaderboard performance.

Suggested report structure:
1. **Problem and related work:** H-alpha filament instance segmentation; distinguish Anomalyzer quality control from the poster's U-Net/MIoU/grouping/ConvNeXt pipeline. Cite SoftwareX, ICDMW, U-Net, and PQ. This notebook is an established-model baseline, not a novel algorithm claim.
2. **Data and method:** annotation version/hash; separate annotator targets; day-grouped split; grayscale normalization; 512-pixel resizing and its limitations; pretrained encoder, Dice+BCE, augmentation, optimizer, epoch/checkpoint selection; connected components and calibrated minimum area in original pixels.
3. **Experiments:** exact tuning candidates, holdout image counts, PQ/Dice/IoU distributions and split/merge relations; U-Net versus contrast on identical splits. Record hardware, seconds and package versions. Additional valid ablations include FPN, resolution/tiling, MIoU/BF1 implementation, geometry grouping, quality-aware weighting and multiple seeds. These are **not implemented or measured claims** here.
4. **Results and limitations:** quote only measured full-run numbers and the actual returned leaderboard score. Include representative and failure overlays, not just the first image. Annotator disagreement, same-day rather than multi-day event grouping, thinning from resizing, nearby-filament merges, intensity confusion with sunspots, and calibration-set selection are limitations. Add confidence intervals/grouped folds before claiming significance. Keep holdout untouched.

Publish this notebook, exact `requirements.txt`, `best_model.pt`, `calibration.json`, `split_manifest.json`, and method documentation in the public code repository; do not publish credentials or redistribute competition images without checking their license. On Kaggle, save a completed notebook version with outputs. The next cell records evidence but does not write a finished paper, invent results, or submit the final form.

In [ ]:
versions = {name: importlib.metadata.version(name) for name in [
    'torch', 'torchvision', 'segmentation-models-pytorch', 'numpy', 'Pillow', 'scipy', 'pycocotools', 'matplotlib']}
with (OUTPUT_DIR / 'requirements.txt').open('w') as stream:
    subprocess.run([sys.executable, '-m', 'pip', 'list', '--format=freeze'], stdout=stream, check=True)
if INPUT_DIR != OUTPUT_DIR:
    for filename in ['best_model.pt', 'calibration.json']:
        shutil.copy2(INPUT_DIR / filename, OUTPUT_DIR / filename)
write_json(OUTPUT_DIR / 'experiment.json', dict(run_name=RUN_NAME, config=dict(CFG),
    smoke_test=SMOKE_TEST, python=sys.version, packages=versions, device=str(DEVICE),
    annotation_sha256=annotation_sha256, holdout=holdout_results, selected_parameters=selected,
    leaderboard_score=None, leaderboard_score_note='Record the actual API/site result after submission.'))
print('Evidence package:', OUTPUT_DIR)
print('Full training and leaderboard results must be obtained on Kaggle; no paper/form was submitted here.')
